# Notebook 02: Preprocessing and Pipeline Construction
## Data Leakage Prevention, Chronological Splitting, and Feature Pipelines

**Key Principles:**
1. **Strict Chronological Splitting:** 80% train, 20% test. Time-series observations are NEVER randomly shuffled.
2. **Fit on Train Only:** Imputers, Scalers, and Encoders are fitted strictly on training data folds.
3. **Model-Specific Scaling:** StandardScaler applied to distance/linear models (LR, KNN, SVM) and omitted for tree ensembles.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_india_dataset
from src.feature_engineering import add_temporal_features
from src.preprocessing import chronological_train_test_split, build_preprocessor

df = load_india_dataset('../data/raw/india_city_day.csv')
df = add_temporal_features(df, date_col='Date')

X_train, X_test, y_train, y_test = chronological_train_test_split(
    df, target_col='AQI_Bucket', train_ratio=0.8, date_col='Date'
)
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')
print(f'y_train distribution:\n{y_train.value_counts()}')

### Pipeline Verification and Zero Data Leakage Test

In [ ]:
cat_cols = ['City', 'season']
num_cols = [c for c in X_train.columns if c not in cat_cols]

prep = build_preprocessor(num_cols, cat_cols, scale=True, impute_strategy='median')
X_train_proc = prep.fit_transform(X_train)
X_test_proc = prep.transform(X_test)

print('Preprocessed X_train shape:', X_train_proc.shape)
print('Preprocessed X_test shape:', X_test_proc.shape)
print('Verified: Preprocessor successfully fitted on X_train only.')